In [1]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

for d in [
    INFERENCE_DIR,
    ARTIFACTS_DIR / "stage1a",
    ARTIFACTS_DIR / "stage2",
    ARTIFACTS_DIR / "stage1c",
]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Clean production package folder created:")
print(BASE_DIR.resolve())

✅ Clean production package folder created:
/mmfs1/home/mohamed.salem/firstJob/pr./FID/STAGE 1 & 2/maintenance_ai_production_clean


In [2]:
from pathlib import Path
import shutil

# =========================================================
# Clean Production Package Structure
# =========================================================

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

STAGE1A_ARTIFACTS_DIR = ARTIFACTS_DIR / "stage1a"
STAGE2_ARTIFACTS_DIR = ARTIFACTS_DIR / "stage2"
STAGE1C_ARTIFACTS_DIR = ARTIFACTS_DIR / "stage1c"

for d in [
    INFERENCE_DIR,
    STAGE1A_ARTIFACTS_DIR,
    STAGE2_ARTIFACTS_DIR,
    STAGE1C_ARTIFACTS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Production package folders created:")
print(BASE_DIR.resolve())


# =========================================================
# Helper: copy required files
# =========================================================

def copy_required_file(src, dst_dir):
    src = Path(src)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print(f"❌ Missing: {src}")
        return False

    dst = dst_dir / src.name
    shutil.copy2(src, dst)
    print(f"✅ Copied: {src} -> {dst}")
    return True


# =========================================================
# Stage 1A Artifacts
# =========================================================

print("\n===== Copying Stage 1A artifacts =====")

stage1a_files = [
    "run_stage1_production/artifacts/stage1_bundle.joblib",
    "run_stage1_production/artifacts/config.json",
]

stage1a_ok = []

for f in stage1a_files:
    stage1a_ok.append(copy_required_file(f, STAGE1A_ARTIFACTS_DIR))


# =========================================================
# Stage 2 Artifacts
# =========================================================

print("\n===== Copying Stage 2 artifacts =====")

stage2_files = [
    "stage2_final_training/stage2_best_model.joblib",
    "stage2_final_training/stage2_word_tfidf.joblib",
    "stage2_final_training/stage2_char_tfidf.joblib",
    "stage2_final_training/stage2_feature_names.joblib",
    "stage2_final_training/stage2_binary_threshold.joblib",
    "stage2_final_training/stage2_accept_threshold.joblib",
    "stage2_final_training/stage2_reject_threshold.joblib",
    "stage2_final_training/stage2_clean_reference_dataset.csv",
]

stage2_ok = []

for f in stage2_files:
    stage2_ok.append(copy_required_file(f, STAGE2_ARTIFACTS_DIR))


# =========================================================
# Stage 1C Artifacts
# =========================================================

print("\n===== Copying Stage 1C artifacts =====")

stage1c_files = [
    "run_stage1c_v31/artifacts/stage1c_v31_bundle.joblib",
    "run_stage1c_v32/artifacts/stage1c_v32_pairwise_bundle.joblib",
]

stage1c_ok = []

for f in stage1c_files:
    stage1c_ok.append(copy_required_file(f, STAGE1C_ARTIFACTS_DIR))


# =========================================================
# Final Check
# =========================================================

print("\n===== COPY SUMMARY =====")
print("Stage 1A:", "✅ OK" if all(stage1a_ok) else "❌ Missing files")
print("Stage 2 :", "✅ OK" if all(stage2_ok) else "❌ Missing files")
print("Stage 1C:", "✅ OK" if all(stage1c_ok) else "❌ Missing files")

print("\n===== Final package tree =====")

for p in sorted(BASE_DIR.rglob("*")):
    if p.is_file():
        print("FILE:", p)
    else:
        print("DIR :", p)

✅ Production package folders created:
/mmfs1/home/mohamed.salem/firstJob/pr./FID/STAGE 1 & 2/maintenance_ai_production_clean

===== Copying Stage 1A artifacts =====
✅ Copied: run_stage1_production/artifacts/stage1_bundle.joblib -> maintenance_ai_production_clean/artifacts/stage1a/stage1_bundle.joblib
✅ Copied: run_stage1_production/artifacts/config.json -> maintenance_ai_production_clean/artifacts/stage1a/config.json

===== Copying Stage 2 artifacts =====
✅ Copied: stage2_final_training/stage2_best_model.joblib -> maintenance_ai_production_clean/artifacts/stage2/stage2_best_model.joblib
✅ Copied: stage2_final_training/stage2_word_tfidf.joblib -> maintenance_ai_production_clean/artifacts/stage2/stage2_word_tfidf.joblib
✅ Copied: stage2_final_training/stage2_char_tfidf.joblib -> maintenance_ai_production_clean/artifacts/stage2/stage2_char_tfidf.joblib
✅ Copied: stage2_final_training/stage2_feature_names.joblib -> maintenance_ai_production_clean/artifacts/stage2/stage2_feature_names.jobli

In [3]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"

code = r'''
# =========================================================
# Stage 1A Production Inference
# Image Relevance Detector
#
# Input:
#   image_path
#
# Output:
#   RELEVANT / IRRELEVANT / UNCERTAIN
# =========================================================

import json
import joblib
import warnings
from pathlib import Path

import numpy as np
from PIL import Image, ImageFile

from scipy.stats import entropy, skew

from skimage.color import rgb2gray, rgb2hsv
from skimage.transform import resize
from skimage.feature import (
    hog,
    local_binary_pattern,
    graycomatrix,
    graycoprops,
    canny
)
from skimage.filters import laplace, sobel


warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True


# =========================================================
# Joblib Required Classes
# =========================================================

class SVMPreprocessor:
    def __init__(self, k_best=4000, pca_components=800):
        self.k_best = k_best
        self.pca_components = pca_components
        self.selector = None
        self.pca = None
        self.scaler = None

    def transform(self, X):
        X_sel = self.selector.transform(X)
        X_pca = self.pca.transform(X_sel)
        X_sc = self.scaler.transform(X_pca)
        return X_sc


class XGBPreprocessor:
    def __init__(self, k_best=5000):
        self.k_best = k_best
        self.selector = None

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# Load
# =========================================================

def load_stage1a(artifacts_dir=None):
    if artifacts_dir is None:
        artifacts_dir = Path(__file__).resolve().parents[1] / "artifacts" / "stage1a"
    else:
        artifacts_dir = Path(artifacts_dir)

    bundle_path = artifacts_dir / "stage1_bundle.joblib"
    config_path = artifacts_dir / "config.json"

    if not bundle_path.exists():
        raise FileNotFoundError(f"Stage 1A bundle not found: {bundle_path}")

    if not config_path.exists():
        raise FileNotFoundError(f"Stage 1A config not found: {config_path}")

    bundle = joblib.load(bundle_path)

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    required_keys = [
        "svm_preprocessor",
        "xgb_preprocessor",
        "svm_model",
        "xgb_model",
        "ensemble_w_xgb",
        "ensemble_w_svm",
        "accept_threshold",
        "reject_threshold",
        "img_size"
    ]

    missing = [k for k in required_keys if k not in bundle]

    if missing:
        raise KeyError(f"Missing keys in Stage 1A bundle: {missing}")

    return bundle, config


# =========================================================
# Feature Extraction
# =========================================================

def extract_stage1a_features(image_path, img_size=(224, 224)):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_pil = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    img = np.array(img_pil)

    img = resize(
        img,
        img_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    if img.max() > 1.0:
        img /= 255.0

    gray = rgb2gray(img)
    hsv = rgb2hsv(img)

    # HOG
    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    # LBP
    lbp = local_binary_pattern(
        gray,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=10,
        range=(0, 10),
        density=True
    )

    # HSV histogram
    color_hist = []

    for ch in range(3):
        hist, _ = np.histogram(
            hsv[:, :, ch],
            bins=32,
            range=(0, 1),
            density=True
        )
        color_hist.extend(hist)

    # HSV color moments
    color_moments = []

    for ch in range(3):
        vals = hsv[:, :, ch].ravel()

        color_moments.extend([
            np.mean(vals),
            np.std(vals),
            skew(vals)
        ])

    # GLCM
    gray_u8 = (gray * 255).astype(np.uint8)

    glcm = graycomatrix(
        gray_u8,
        distances=[1, 2],
        angles=[0, np.pi / 4, np.pi / 2],
        levels=256,
        symmetric=True,
        normed=True
    )

    glcm_feat = []

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM"
    ]:
        glcm_feat.extend(graycoprops(glcm, prop).ravel())

    # Edge / sharpness
    edges = canny(gray)
    edge_density = edges.mean()

    sob = sobel(gray)

    sobel_hist, _ = np.histogram(
        sob.ravel(),
        bins=16,
        range=(0, 1),
        density=True
    )

    lap_var = laplace(gray).var()

    # Global stats
    gray_hist, _ = np.histogram(
        gray.ravel(),
        bins=64,
        range=(0, 1),
        density=True
    )

    ent = entropy(gray_hist + 1e-8)

    aspect_ratio = orig_w / (orig_h + 1e-8)
    brightness_mean = gray.mean()
    brightness_std = gray.std()

    global_feat = np.array([
        edge_density,
        lap_var,
        ent,
        aspect_ratio,
        brightness_mean,
        brightness_std,
        orig_w,
        orig_h
    ], dtype=np.float32)

    feat = np.concatenate([
        hog_feat.astype(np.float32),
        lbp_hist.astype(np.float32),
        np.array(color_hist, dtype=np.float32),
        np.array(color_moments, dtype=np.float32),
        np.array(glcm_feat, dtype=np.float32),
        sobel_hist.astype(np.float32),
        global_feat
    ]).astype(np.float32)

    feat = np.nan_to_num(
        feat,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return feat


# =========================================================
# Prediction
# =========================================================

def decide_stage1a(prob_relevant, accept_threshold, reject_threshold):
    if prob_relevant >= accept_threshold:
        return "RELEVANT"

    if prob_relevant <= reject_threshold:
        return "IRRELEVANT"

    return "UNCERTAIN"


def predict_stage1a_image(
    image_path,
    stage1_bundle,
    deploy_reject_threshold=0.15,
    use_deploy_reject_threshold=True
):
    img_size = tuple(stage1_bundle.get("img_size", (224, 224)))

    features = extract_stage1a_features(
        image_path=image_path,
        img_size=img_size
    )

    X = features.reshape(1, -1)

    svm_preprocessor = stage1_bundle["svm_preprocessor"]
    xgb_preprocessor = stage1_bundle["xgb_preprocessor"]

    svm_model = stage1_bundle["svm_model"]
    xgb_model = stage1_bundle["xgb_model"]

    ensemble_w_xgb = float(stage1_bundle.get("ensemble_w_xgb", 0.75))
    ensemble_w_svm = float(stage1_bundle.get("ensemble_w_svm", 0.25))

    accept_threshold = float(stage1_bundle["accept_threshold"])

    if use_deploy_reject_threshold:
        reject_threshold = float(deploy_reject_threshold)
    else:
        reject_threshold = float(stage1_bundle["reject_threshold"])

    X_xgb = xgb_preprocessor.transform(X)
    X_svm = svm_preprocessor.transform(X)

    prob_xgb = float(xgb_model.predict_proba(X_xgb)[0, 1])
    prob_svm = float(svm_model.predict_proba(X_svm)[0, 1])

    prob_relevant = float(
        ensemble_w_xgb * prob_xgb
        +
        ensemble_w_svm * prob_svm
    )

    decision = decide_stage1a(
        prob_relevant=prob_relevant,
        accept_threshold=accept_threshold,
        reject_threshold=reject_threshold
    )

    return {
        "stage": "Stage 1A",
        "image_path": str(image_path),
        "decision": decision,
        "prob_relevant": prob_relevant,
        "prob_xgb": prob_xgb,
        "prob_svm": prob_svm,
        "accept_threshold": accept_threshold,
        "reject_threshold": reject_threshold
    }
'''

output_path = INFERENCE_DIR / "stage1a_inference.py"
output_path.write_text(code, encoding="utf-8")

print("✅ Saved:", output_path)
print("Exists:", output_path.exists())

✅ Saved: maintenance_ai_production_clean/inference/stage1a_inference.py
Exists: True


In [4]:
import py_compile
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/stage1a_inference.py")

py_compile.compile(str(path), doraise=True)
print("✅ stage1a_inference.py syntax OK")

✅ stage1a_inference.py syntax OK


In [5]:
import sys
import importlib
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())
if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

from inference.stage1a_inference import load_stage1a, predict_stage1a_image

bundle, config = load_stage1a()
print("✅ Stage 1A loaded from clean package")
print("Bundle keys:", list(bundle.keys()))

Dataset size: 5026
label
0    3021
1    2005
Name: count, dtype: int64

Extracting features...


100%|██████████| 5026/5026 [09:47<00:00,  8.55it/s]


Feature matrix: (5026, 26419)
Failed images: 0

Hold-out split:
Train Full: (4020, 26419) [2416 1604]
Test      : (1006, 26419) [605 401]

K-FOLD CROSS VALIDATION START

===== FOLD 1/5 =====
Hard negatives found: 14
Accept threshold: 0.3089
Reject threshold: 0.0753
Accuracy        : 0.9552
Precision       : 0.9524
Recall          : 0.9346
F1              : 0.9434
ROC AUC         : 0.9903
PR AUC          : 0.9878
Coverage        : 0.8781

===== FOLD 2/5 =====
Hard negatives found: 15
Accept threshold: 0.2446
Reject threshold: 0.0723
Accuracy        : 0.9602
Precision       : 0.9502
Recall          : 0.9502
F1              : 0.9502
ROC AUC         : 0.9834
PR AUC          : 0.9847
Coverage        : 0.8918

===== FOLD 3/5 =====
Hard negatives found: 14
Accept threshold: 0.3627
Reject threshold: 0.0753
Accuracy        : 0.9490
Precision       : 0.9516
Recall          : 0.9190
F1              : 0.9350
ROC AUC         : 0.9863
PR AUC          : 0.9858
Coverage        : 0.8744

===== FOLD 4/5

In [6]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"

code = r'''
# =========================================================
# Stage 2 Production Inference
# Arabic Text Category/Subcategory Matcher
#
# Input:
#   Arabic user description
#
# Output:
#   MATCH / MISMATCH
#   category
#   subcategory
#   confidence
# =========================================================

import re
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


warnings.filterwarnings("ignore")


# =========================================================
# Config
# =========================================================

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

FINAL_MATCH_MIN_SCORE = 0.42
FINAL_MATCH_MIN_MARGIN = 0.035
FINAL_MATCH_MIN_RULE = 0.25


# =========================================================
# Manual Production Prototypes
# =========================================================

DEFAULT_MANUAL_PROTOTYPES = {
    "تسريب مياه": {
        "category": "سباكة",
        "prototype_text": "تسريب مياه حنفية صنبور ماسورة مواسير بلل تنقيط رطوبة مياه",
        "keywords": "تسريب|تسرب|مياه|ميه|ماء|حنفية|حنفيه|صنبور|ماسورة|ماسوره|مواسير|بلل|تنقيط|رطوبة|رطوبه"
    },
    "عطل صرف": {
        "category": "سباكة",
        "prototype_text": "انسداد صرف حوض بلاعة مياه لا تنزل المياه متجمعة مسدود",
        "keywords": "انسداد|صرف|حوض|بلاعة|بلاعه|مياه|ميه|مسدود|متجمعة|متجمعه|مجاري|تسليك"
    },
    "عطل مرحاض": {
        "category": "سباكة",
        "prototype_text": "عطل مرحاض تواليت قاعدة حمام سيفون انسداد المرحاض لا يسحب المياه",
        "keywords": "مرحاض|تواليت|قاعدة|قاعده|حمام|سيفون|انسداد|مياه|ميه|لا يسحب|مش بيسحب"
    },
    "عطل حوض": {
        "category": "سباكة",
        "prototype_text": "عطل حوض تسريب حوض انسداد حوض مياه تحت الحوض صرف الحوض",
        "keywords": "حوض|الحوض|تسريب|انسداد|صرف|مياه|ميه|تحت|مسدود|خلاط"
    },
    "عطل سباكة عام": {
        "category": "سباكة",
        "prototype_text": "عطل سباكة مشكلة مياه مواسير حمام مطبخ سباك تصليح سباكة",
        "keywords": "سباكة|سباكه|سباك|مياه|ميه|مواسير|حمام|مطبخ|عطل|مشكلة|مشكله|تصليح"
    },
    "عطل كهرباء عام": {
        "category": "كهرباء",
        "prototype_text": "عطل كهرباء نور إضاءة فيشة مقبس سلك شرار ماس كهرباء",
        "keywords": "كهرباء|كهربا|نور|إضاءة|اضاءة|اضاءه|لمبة|لمبه|لمبات|فيشة|فيشه|مقبس|بريزة|بريزه|سلك|أسلاك|اسلاك|شرار|ماس|بيقطع|بيفصل"
    },
    "تشققات جدار": {
        "category": "نقاشة",
        "prototype_text": "تشققات جدار شروخ حائط حيطة شرخ في الجدار تشقق واضح",
        "keywords": "تشققات|تشقق|جدار|شروخ|شرخ|حائط|حيطة|حيطه|مشققة|مشققه"
    },
    "تلف دهان": {
        "category": "نقاشة",
        "prototype_text": "تلف دهان طلاء مقشر تقشير دهان بقع رطوبة الحائط",
        "keywords": "دهان|طلاء|بويه|بوية|تقشير|مقشر|متقشر|بقع|بقعة|بقعه|رطوبة|رطوبه|حائط|حيطة|حيطه|تلف"
    },
    "تلف سقف": {
        "category": "نقاشة",
        "prototype_text": "تلف سقف رطوبة سقف دهان واقع من السقف آثار مياه بقع",
        "keywords": "سقف|رطوبة|رطوبه|دهان|مياه|ميه|تلف|بقع|آثار|اثار|نشع"
    },
    "عطل نقاشة عام": {
        "category": "نقاشة",
        "prototype_text": "عطل نقاشة مشكلة دهان حائط جدار تشطيب نقاش تصليح نقاشة",
        "keywords": "نقاشة|نقاشه|نقاش|دهان|حائط|حيطة|حيطه|جدار|تشطيب|عطل|مشكلة|مشكله|تصليح|محارة|محاره"
    },
    "عطل باب": {
        "category": "نجارة",
        "prototype_text": "عطل باب خشب الباب لا يغلق مفصلة الباب مكسور بيحك نجارة",
        "keywords": "باب|خشب|يغلق|يقفل|بيقفل|مش بيقفل|مفصلة|مفصله|مكسور|بيحك|نجارة|نجاره|كالون|مقبض"
    },
    "عطل خشب": {
        "category": "نجارة",
        "prototype_text": "عطل خشب شباك خشب درفة مطبخ إطار خشبي مكسور تلف",
        "keywords": "خشب|شباك|نافذة|نافذه|درفة|درفه|ضلفة|ضلفه|مطبخ|إطار|اطار|مكسور|تلف"
    },
    "عطل نجارة عام": {
        "category": "نجارة",
        "prototype_text": "عطل نجارة مشكلة خشب تصليح نجار أثاث خشبي مفكوك",
        "keywords": "نجارة|نجاره|نجار|خشب|تصليح|أثاث|اثاث|خشبي|مفكوك|عطل|مشكلة|مشكله"
    }
}


# =========================================================
# Text Helpers
# =========================================================

ARABIC_STOPWORDS = set([
    "في", "من", "على", "مع", "هناك", "حيث", "هذه", "هذا",
    "تظهر", "يظهر", "صورة", "الصورة", "العناصر", "الظاهرة",
    "داخل", "إلى", "الى", "أن", "ان", "بما", "كما", "عن",
    "و", "أو", "او", "هو", "هي", "كان", "كانت", "تم", "قد",
    "كل", "أي", "اي", "دي", "ده", "دا"
])


def normalize_text(text):
    text = str(text)

    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("آ", "ا")
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    text = normalize_text(text)
    words = text.split()

    return [
        w for w in words
        if w not in ARABIC_STOPWORDS and len(w) > 1
    ]


def keyword_set(k):
    words = str(k).split("|")
    words = [normalize_text(w) for w in words]
    words = [w for w in words if len(w) > 1]
    return set(words)


def jaccard(s1, s2):
    if not s1 or not s2:
        return 0.0

    return len(s1 & s2) / len(s1 | s2)


def containment(s1, s2):
    if not s1 or not s2:
        return 0.0

    return len(s1 & s2) / min(len(s1), len(s2))


def length_features(t1, t2):
    len1 = len(str(t1))
    len2 = len(str(t2))

    diff = abs(len1 - len2)
    ratio = min(len1, len2) / max(len1, len2) if max(len1, len2) > 0 else 0.0

    return diff, ratio


def damage_terms_overlap(t1, t2):
    important_terms = [
        "تسريب", "تسرب", "مياه", "ميه", "ماء",
        "حنفيه", "حنفية", "صنبور", "حوض", "صرف", "مجاري",
        "مرحاض", "تواليت", "انسداد", "مسدود", "ماسوره", "ماسورة",
        "كهرباء", "كهربا", "نور", "اضاءه", "اضاءة", "لمبه", "لمبة",
        "اسلاك", "أسلاك", "سلك", "شرار", "ماس", "فيشه", "فيشة", "مقبس",
        "جدار", "حيطه", "حيطة", "حائط", "تشققات", "تشقق", "شروخ", "شرخ",
        "دهان", "طلاء", "سقف", "رطوبه", "رطوبة", "بلل", "بقع",
        "باب", "شباك", "نافذه", "نافذة", "خشب", "درفه", "درفة",
        "عطل", "تلف", "مشكله", "مشكلة", "خلل", "كسر", "مكسور"
    ]

    t1 = normalize_text(t1)
    t2 = normalize_text(t2)

    s1 = set([normalize_text(term) for term in important_terms if normalize_text(term) in t1])
    s2 = set([normalize_text(term) for term in important_terms if normalize_text(term) in t2])

    return jaccard(s1, s2)


def cosine_dense(v1, v2):
    v1 = np.asarray(v1).reshape(1, -1)
    v2 = np.asarray(v2).reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0, 0])


def cosine_sparse(v1, v2):
    return float(cosine_similarity(v1, v2)[0, 0])


# =========================================================
# Rule-based Score
# =========================================================

def subcategory_rule_score(user_text, prototype):
    text = normalize_text(user_text)
    subcategory = prototype["subcategory"]
    category = prototype["category"]

    keywords = keyword_set(prototype.get("keywords", ""))
    hits = [kw for kw in keywords if kw and kw in text]
    keyword_score = min(len(hits) / 3.0, 1.0)

    category_terms = {
        "سباكة": [
            "سباكه", "سباكة", "سباك", "مواسير", "ماسوره", "ماسورة",
            "مياه", "ميه", "ماء", "حنفيه", "حنفية", "صنبور", "خلاط",
            "حوض", "صرف", "بلاعه", "بلاعة", "مجاري", "انسداد", "مسدود",
            "مرحاض", "تواليت", "قاعدة", "قاعده", "حمام", "سيفون",
            "تسريب", "تسرب", "بتسرب", "تنقيط", "بتنقط", "نقطه", "نقطة",
            "بلل", "رطوبه", "رطوبة", "تحت الحوض", "المياه مش بتنزل",
            "الميه مش بتنزل", "المياه لا تنزل", "الميه لا تنزل",
            "مش بيسحب", "لا يسحب", "بيطفح", "طفح", "تسليك"
        ],
        "كهرباء": [
            "كهرباء", "كهربا", "كهربائي", "كهربائى",
            "نور", "انوار", "أنوار", "اضاءه", "اضاءة", "إضاءة",
            "لمبه", "لمبة", "لمبات", "مصباح", "مصابيح",
            "فيشه", "فيشة", "مقبس", "بريزه", "بريزة", "مشترك",
            "سلك", "اسلاك", "أسلاك", "كابل", "كابلات",
            "شرار", "شرز", "ماس", "قفله", "قفلة", "شورت",
            "فاصل", "فاصله", "فاصلة", "بيفصل", "بتفصل",
            "بيقطع", "بتقطع", "قطع كهرباء", "الكهرباء قاطعه",
            "الكهربا قاطعه", "قاطع", "لوحه", "لوحة", "عداد",
            "مفتاح", "مفتاح نور", "زر", "فيش", "محروق", "ريحة شياط",
            "سخونه", "سخونة", "مروحه", "مروحة", "شفاط"
        ],
        "نقاشة": [
            "نقاشه", "نقاشة", "نقاش", "دهان", "دهانات",
            "طلاء", "بويه", "بوية", "تشطيب", "محاره", "محارة",
            "حائط", "حيطه", "حيطة", "جدار", "جدران", "حوايط",
            "شرخ", "شروخ", "تشققات", "تشقق", "مشققه", "مشققة",
            "سقف", "اسقف", "أسقف", "رطوبه", "رطوبة", "نشع",
            "بقع", "بقعه", "بقعة", "تقشير", "مقشر", "مقشره", "مقشرة",
            "واقع", "واقعه", "واقعة", "متقشر", "متقشرة",
            "اثار مياه", "آثار مياه", "تساقط دهان", "تلف دهان",
            "لون", "اللون", "بهتان", "متعفن", "عفن"
        ],
        "نجارة": [
            "نجاره", "نجارة", "نجار", "خشب", "خشبي", "خشبى",
            "باب", "ابواب", "أبواب", "شباك", "شبابيك", "نافذه", "نافذة",
            "درفه", "درفة", "ضلفه", "ضلفة", "دولاب", "مطبخ",
            "درج", "ادراج", "أدراج", "رف", "ارفف", "أرفف",
            "مفصله", "مفصلة", "مفصلات", "كالون", "مقبض", "اكره", "أكرة",
            "مكسور", "كسر", "مخلوع", "مفكوك", "متفكك", "بيحك",
            "بيقفل", "يقفل", "مش بيقفل", "مش بيتقفل", "لا يغلق",
            "يفتح", "مش بيفتح", "متاكل", "متآكل", "تآكل",
            "اطار", "إطار", "برواز", "اثاث", "أثاث", "ترابيزة", "كرسي"
        ]
    }

    subcategory_terms = {
        "تسريب مياه": [
            "تسريب", "تسرب", "بتسرب", "بيسرب", "تسريب مياه", "تسريب ميه",
            "تنقيط", "بتنقط", "بينقط", "نقطه", "نقطة", "مياه بتنزل",
            "ميه بتنزل", "مياه سايبه", "ميه سايبه",
            "حنفيه", "حنفية", "صنبور", "خلاط", "ماسوره", "ماسورة",
            "مواسير", "بلل", "رطوبه", "رطوبة", "تحت الحوض", "جنب الحوض"
        ],
        "عطل صرف": [
            "صرف", "انسداد", "مسدود", "بلاعه", "بلاعة", "مجاري",
            "الحوض مسدود", "الصرف مسدود", "صرف الحمام", "صرف المطبخ",
            "المياه مش بتنزل", "الميه مش بتنزل", "مياه مش بتنزل", "ميه مش بتنزل",
            "المياه لا تنزل", "الميه لا تنزل", "المياه متجمعه", "المياه متجمعة",
            "الميه متجمعه", "الميه متجمعة", "تسليك", "طفح", "بيطفح"
        ],
        "عطل مرحاض": [
            "مرحاض", "تواليت", "قاعدة", "قاعده", "حمام", "سيفون",
            "المرحاض مسدود", "التواليت مسدود", "مش بيسحب", "لا يسحب",
            "مش بيطرد", "السيفون مش شغال", "السيفون بايظ",
            "مياه المرحاض", "ميه المرحاض", "قاعدة الحمام", "قاعده الحمام"
        ],
        "عطل حوض": [
            "حوض", "الحوض", "تحت الحوض", "صرف الحوض", "الحوض مسدود",
            "الحوض بيسرب", "الحوض بينقط", "مياه تحت الحوض", "ميه تحت الحوض",
            "خلاط الحوض", "حنفية الحوض", "حنفيه الحوض"
        ],
        "عطل سباكة عام": [
            "سباك", "سباكه", "سباكة", "مشكلة سباكة", "مشكله سباكه",
            "عطل سباكة", "عطل سباكه", "مواسير", "مياه", "ميه",
            "حمام", "مطبخ", "تصليح سباكة", "تصليح سباكه", "محتاج سباك"
        ],
        "عطل كهرباء عام": [
            "كهرباء", "كهربا", "النور", "نور", "اضاءه", "اضاءة", "إضاءة",
            "لمبه", "لمبة", "لمبات", "مصباح", "فيشه", "فيشة", "مقبس",
            "بريزه", "بريزة", "سلك", "اسلاك", "أسلاك", "كابل",
            "شرار", "ماس", "قفله", "قفلة", "شورت", "بيفصل", "بتفصل",
            "بيقطع", "بتقطع", "الكهرباء فاصله", "الكهربا فاصله",
            "الكهرباء قاطعه", "الكهربا قاطعه", "مفتاح نور", "محروق"
        ],
        "تشققات جدار": [
            "شرخ", "شروخ", "تشققات", "تشقق", "شق", "حيطه", "حيطة",
            "حائط", "جدار", "الجدار", "الحائط", "الحيطه", "الحيطة",
            "مشققه", "مشققة", "شرخ طويل", "شروخ واضحه", "شروخ واضحة"
        ],
        "تلف دهان": [
            "دهان", "طلاء", "بويه", "بوية", "تقشير", "مقشر", "متقشر",
            "الدهان مقشر", "الطلاء واقع", "الدهان واقع", "بقع", "بقعه",
            "بقعة", "تلف دهان", "لون", "بهتان", "الحائط مقشر"
        ],
        "تلف سقف": [
            "سقف", "السقف", "رطوبه", "رطوبة", "نشع", "بقع سقف",
            "بقع في السقف", "اثار مياه", "آثار مياه", "دهان واقع من السقف",
            "السقف فيه رطوبة", "السقف فيه رطوبه", "السقف فيه بقع",
            "مياه في السقف", "ميه في السقف"
        ],
        "عطل نقاشة عام": [
            "نقاش", "نقاشه", "نقاشة", "مشكلة نقاشة", "مشكله نقاشه",
            "عطل نقاشة", "عطل نقاشه", "دهان", "حائط", "حيطه", "حيطة",
            "جدار", "تشطيب", "محاره", "محارة", "تصليح حائط", "محتاج نقاش"
        ],
        "عطل باب": [
            "باب", "الباب", "مش بيقفل", "مش بيتقفل", "بيقفل", "يقفل",
            "لا يغلق", "مفصله", "مفصلة", "مفصلات", "مقبض", "كالون",
            "اكره", "أكرة", "بيحك", "مخلوع", "الباب مكسور", "الباب بايظ"
        ],
        "عطل خشب": [
            "خشب", "خشبي", "خشبى", "شباك", "الشباك", "نافذه", "نافذة",
            "درفه", "درفة", "ضلفه", "ضلفة", "مطبخ", "دولاب", "اطار",
            "إطار", "مكسور", "متاكل", "متآكل", "تالف", "الخشب مكسور",
            "الشباك الخشب مكسور"
        ],
        "عطل نجارة عام": [
            "نجار", "نجاره", "نجارة", "محتاج نجار", "تصليح نجارة",
            "تصليح نجاره", "خشب", "اثاث", "أثاث", "مفكوك", "متفكك",
            "مشكلة نجارة", "مشكله نجاره", "عطل نجارة", "عطل نجاره",
            "حاجة خشب", "حاجه خشب"
        ]
    }

    cat_hits = sum(
        1 for term in category_terms.get(category, [])
        if normalize_text(term) in text
    )

    sub_hits = sum(
        1 for term in subcategory_terms.get(subcategory, [])
        if normalize_text(term) in text
    )

    category_score = min(cat_hits / 2.0, 1.0)
    subcategory_score = min(sub_hits / 2.0, 1.0)

    irrelevant_terms = [
        "عربيه", "عربية", "سياره", "سيارة", "شارع", "اكل", "أكل",
        "قطة", "قطه", "كلب", "شخص", "موبايل", "لابتوب", "ملابس",
        "شجره", "شجرة", "طريق", "بحر", "سما", "سماء"
    ]

    irrelevant_hit = any(normalize_text(term) in text for term in irrelevant_terms)

    final_rule_score = (
        0.45 * subcategory_score
        +
        0.35 * category_score
        +
        0.20 * keyword_score
    )

    if irrelevant_hit and final_rule_score < 0.45:
        final_rule_score *= 0.25

    return float(final_rule_score)


# =========================================================
# Load Stage 2
# =========================================================

def load_stage2(
    artifacts_dir=None,
    prototypes_path=None,
    embedding_model_name=EMBEDDING_MODEL_NAME
):
    if artifacts_dir is None:
        artifacts_dir = Path(__file__).resolve().parents[1] / "artifacts" / "stage2"
    else:
        artifacts_dir = Path(artifacts_dir)

    required_files = {
        "model": artifacts_dir / "stage2_best_model.joblib",
        "word_tfidf": artifacts_dir / "stage2_word_tfidf.joblib",
        "char_tfidf": artifacts_dir / "stage2_char_tfidf.joblib",
        "feature_names": artifacts_dir / "stage2_feature_names.joblib",
        "binary_threshold": artifacts_dir / "stage2_binary_threshold.joblib",
        "accept_threshold": artifacts_dir / "stage2_accept_threshold.joblib",
        "reject_threshold": artifacts_dir / "stage2_reject_threshold.joblib",
        "reference_dataset": artifacts_dir / "stage2_clean_reference_dataset.csv"
    }

    missing_files = []

    for name, path in required_files.items():
        if not path.exists():
            missing_files.append((name, str(path)))

    if missing_files:
        message = "Missing Stage 2 files:\n"
        for name, path in missing_files:
            message += f"- {name}: {path}\n"
        raise FileNotFoundError(message)

    model = joblib.load(required_files["model"])
    word_tfidf = joblib.load(required_files["word_tfidf"])
    char_tfidf = joblib.load(required_files["char_tfidf"])
    feature_names = joblib.load(required_files["feature_names"])

    binary_threshold = float(joblib.load(required_files["binary_threshold"]))
    accept_threshold = float(joblib.load(required_files["accept_threshold"]))
    reject_threshold = float(joblib.load(required_files["reject_threshold"]))

    reference_df = pd.read_csv(
        required_files["reference_dataset"],
        encoding="utf-8-sig"
    )

    prototypes = load_stage2_prototypes(prototypes_path)

    embedder = SentenceTransformer(embedding_model_name)

    stage2 = {
        "model": model,
        "word_tfidf": word_tfidf,
        "char_tfidf": char_tfidf,
        "feature_names": feature_names,
        "binary_threshold": binary_threshold,
        "accept_threshold": accept_threshold,
        "reject_threshold": reject_threshold,
        "reference_df": reference_df,
        "prototypes": prototypes,
        "embedder": embedder,
        "embedding_model_name": embedding_model_name
    }

    prepare_stage2_prototypes(stage2)
    validate_stage2(stage2)

    return stage2


def load_stage2_prototypes(prototypes_path=None):
    if prototypes_path is not None:
        prototypes_path = Path(prototypes_path)

        if prototypes_path.exists():
            with open(prototypes_path, "r", encoding="utf-8") as f:
                return json.load(f)

    return DEFAULT_MANUAL_PROTOTYPES.copy()


def prepare_stage2_prototypes(stage2):
    embedder = stage2["embedder"]
    word_tfidf = stage2["word_tfidf"]
    char_tfidf = stage2["char_tfidf"]

    for subcategory, proto in stage2["prototypes"].items():
        proto_text = normalize_text(proto["prototype_text"])

        proto["subcategory"] = subcategory
        proto["normalized_text"] = proto_text
        proto["word_vector"] = word_tfidf.transform([proto_text])
        proto["char_vector"] = char_tfidf.transform([proto_text])

        proto_embedding = embedder.encode(
            proto_text,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        proto["embedding"] = np.asarray(proto_embedding, dtype=np.float32)


def validate_stage2(stage2):
    expected_features = [
        "minilm_sim",
        "word_tfidf_sim",
        "char_tfidf_sim",
        "keyword_sim",
        "word_jaccard",
        "word_containment",
        "domain_overlap",
        "len_diff",
        "len_ratio",
        "cat_match",
        "semantic_pair_sim",
        "semantic_lexical_gap",
        "semantic_keyword_product",
        "lexical_domain_product"
    ]

    loaded_features = list(stage2["feature_names"])
    missing = [f for f in expected_features if f not in loaded_features]

    if missing:
        raise ValueError(f"Stage 2 feature names are missing: {missing}")

    if len(stage2["prototypes"]) == 0:
        raise ValueError("No Stage 2 prototypes found.")


# =========================================================
# Feature Extraction
# =========================================================

def build_stage2_pair_features(user_text, prototype, stage2, user_embedding=None):
    feature_names = list(stage2["feature_names"])

    text1 = normalize_text(user_text)
    text2 = prototype["normalized_text"]

    if user_embedding is None:
        user_embedding = stage2["embedder"].encode(
            text1,
            normalize_embeddings=True,
            show_progress_bar=False
        )

    user_embedding = np.asarray(user_embedding, dtype=np.float32)

    minilm_sim = cosine_dense(user_embedding, prototype["embedding"])

    word_vec_1 = stage2["word_tfidf"].transform([text1])
    word_tfidf_sim = cosine_sparse(word_vec_1, prototype["word_vector"])

    char_vec_1 = stage2["char_tfidf"].transform([text1])
    char_tfidf_sim = cosine_sparse(char_vec_1, prototype["char_vector"])

    kw1 = "|".join(tokenize(text1))
    kw2 = prototype.get("keywords", "")

    kw_set_1 = keyword_set(kw1)
    kw_set_2 = keyword_set(kw2)

    keyword_sim = jaccard(kw_set_1, kw_set_2)

    words_1 = set(tokenize(text1))
    words_2 = set(tokenize(text2))

    word_jaccard = jaccard(words_1, words_2)
    word_containment = containment(words_1, words_2)

    domain_overlap = damage_terms_overlap(text1, text2)

    len_diff, len_ratio = length_features(text1, text2)

    cat_match = 1.0

    semantic_pair_sim = minilm_sim
    semantic_lexical_gap = abs(minilm_sim - word_tfidf_sim)
    semantic_keyword_product = minilm_sim * keyword_sim
    lexical_domain_product = word_jaccard * domain_overlap

    features_dict = {
        "minilm_sim": minilm_sim,
        "word_tfidf_sim": word_tfidf_sim,
        "char_tfidf_sim": char_tfidf_sim,
        "keyword_sim": keyword_sim,
        "word_jaccard": word_jaccard,
        "word_containment": word_containment,
        "domain_overlap": domain_overlap,
        "len_diff": len_diff,
        "len_ratio": len_ratio,
        "cat_match": cat_match,
        "semantic_pair_sim": semantic_pair_sim,
        "semantic_lexical_gap": semantic_lexical_gap,
        "semantic_keyword_product": semantic_keyword_product,
        "lexical_domain_product": lexical_domain_product
    }

    X = np.array(
        [[features_dict[name] for name in feature_names]],
        dtype=np.float32
    )

    return X, features_dict


# =========================================================
# Model Probability
# =========================================================

def get_model_probability(model_artifact, X):
    if isinstance(model_artifact, dict):
        if "trained_stack_models" not in model_artifact:
            raise ValueError(
                "Stage 2 model artifact is a dict but does not contain 'trained_stack_models'."
            )

        trained_models = model_artifact["trained_stack_models"]
        weights = np.asarray(model_artifact.get("weights", None), dtype=np.float64)

        if isinstance(trained_models, dict):
            model_names = list(trained_models.keys())
            models = [trained_models[name] for name in model_names]
        else:
            models = list(trained_models)
            model_names = model_artifact.get(
                "stacking_model_names",
                [f"model_{i}" for i in range(len(models))]
            )

        if weights is None or len(weights) != len(models):
            weights = np.ones(len(models), dtype=np.float64) / len(models)
        else:
            weights = weights / max(weights.sum(), 1e-12)

        probs = []

        for name, model in zip(model_names, models):
            if hasattr(model, "predict_proba"):
                p = float(model.predict_proba(X)[0, 1])
            elif hasattr(model, "decision_function"):
                score = float(model.decision_function(X)[0])
                p = float(1.0 / (1.0 + np.exp(-score)))
            else:
                p = float(model.predict(X)[0])

            probs.append(p)

        probs = np.asarray(probs, dtype=np.float64)
        return float(np.sum(weights * probs))

    model = model_artifact

    if hasattr(model, "predict_proba"):
        return float(model.predict_proba(X)[0, 1])

    if hasattr(model, "decision_function"):
        score = float(model.decision_function(X)[0])
        return float(1.0 / (1.0 + np.exp(-score)))

    return float(model.predict(X)[0])


# =========================================================
# Ranking and Prediction
# =========================================================

def rank_stage2_prototypes(user_text, stage2):
    text = normalize_text(user_text)

    user_embedding = stage2["embedder"].encode(
        text,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    rows = []

    for subcategory, prototype in stage2["prototypes"].items():
        X, features_dict = build_stage2_pair_features(
            user_text=text,
            prototype=prototype,
            stage2=stage2,
            user_embedding=user_embedding
        )

        model_prob = get_model_probability(stage2["model"], X)
        rule_score = subcategory_rule_score(text, prototype)

        minilm_sim = features_dict["minilm_sim"]
        word_tfidf_sim = features_dict["word_tfidf_sim"]
        char_tfidf_sim = features_dict["char_tfidf_sim"]
        keyword_sim = features_dict["keyword_sim"]
        domain_overlap = features_dict["domain_overlap"]

        final_score = (
            0.30 * model_prob
            +
            0.25 * rule_score
            +
            0.20 * minilm_sim
            +
            0.10 * word_tfidf_sim
            +
            0.05 * char_tfidf_sim
            +
            0.05 * keyword_sim
            +
            0.05 * domain_overlap
        )

        rows.append({
            "category": prototype["category"],
            "subcategory": subcategory,
            "prob": float(model_prob),
            "rule_score": float(rule_score),
            "final_score": float(final_score),
            "features": features_dict
        })

    return sorted(rows, key=lambda r: r["final_score"], reverse=True)


def decide_stage2(best_score, margin, best_rule_score):
    if best_score >= 0.50 and best_rule_score >= 0.80:
        return "MATCH"

    if (
        best_score >= FINAL_MATCH_MIN_SCORE
        and margin >= FINAL_MATCH_MIN_MARGIN
        and best_rule_score >= FINAL_MATCH_MIN_RULE
    ):
        return "MATCH"

    return "MISMATCH"


def predict_stage2_text(user_text, stage2, top_k=5):
    if user_text is None or len(str(user_text).strip()) == 0:
        return {
            "stage": "Stage 2",
            "input_text": user_text,
            "decision": "MISMATCH",
            "category": None,
            "subcategory": None,
            "confidence": 0.0,
            "model_confidence": 0.0,
            "rule_score": 0.0,
            "margin": 0.0,
            "reason": "empty_text",
            "top_candidates": []
        }

    ranked = rank_stage2_prototypes(user_text, stage2)

    best = ranked[0]
    second = ranked[1] if len(ranked) > 1 else None

    best_score = float(best["final_score"])
    second_score = float(second["final_score"]) if second else 0.0
    margin = best_score - second_score

    decision = decide_stage2(
        best_score=best_score,
        margin=margin,
        best_rule_score=float(best["rule_score"])
    )

    return {
        "stage": "Stage 2",
        "input_text": user_text,
        "decision": decision,
        "category": best["category"],
        "subcategory": best["subcategory"],
        "confidence": best_score,
        "model_confidence": float(best["prob"]),
        "rule_score": float(best["rule_score"]),
        "margin": margin,
        "top_candidates": [
            {
                "category": row["category"],
                "subcategory": row["subcategory"],
                "prob": float(row["final_score"]),
                "model_prob": float(row["prob"]),
                "rule_score": float(row["rule_score"])
            }
            for row in ranked[:top_k]
        ]
    }
'''

output_path = INFERENCE_DIR / "stage2_inference.py"
output_path.write_text(code, encoding="utf-8")

print("✅ Saved:", output_path)
print("Exists:", output_path.exists())

✅ Saved: maintenance_ai_production_clean/inference/stage2_inference.py
Exists: True


In [7]:
import py_compile
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/stage2_inference.py")

py_compile.compile(str(path), doraise=True)
print("✅ stage2_inference.py syntax OK")

✅ stage2_inference.py syntax OK


In [8]:
from pathlib import Path

text = Path("maintenance_ai_production_clean/inference/stage2_inference.py").read_text(encoding="utf-8")

print("has load_stage2:", "def load_stage2" in text)
print("has predict_stage2_text:", "def predict_stage2_text" in text)
print("has training pipeline:", "FINAL TRAINING PIPELINE" in text)
print("has semantic pair generation:", "SEMANTIC PAIR GENERATION" in text)
print("has WOA training:", "WOA Optimized" in text or "WOA OPTIMIZED" in text)

has load_stage2: True
has predict_stage2_text: True
has training pipeline: False
has semantic pair generation: False
has WOA training: False


In [9]:
import sys
import importlib
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())
if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

if "inference.stage2_inference" in sys.modules:
    del sys.modules["inference.stage2_inference"]

importlib.invalidate_caches()

from inference.stage2_inference import load_stage2, predict_stage2_text

stage2 = load_stage2()
print("✅ Stage 2 loaded from clean package")

result = predict_stage2_text("الحنفية بتسرب ميه طول الوقت", stage2)

print(result["decision"])
print(result["category"])
print(result["subcategory"])
print(round(result["confidence"], 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Stage 2 loaded from clean package
MATCH
سباكة
تسريب مياه
0.4985


In [10]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"

code = r'''
# =========================================================
# Stage 1C V3.1 Feature Extraction
#
# Handcrafted visual features used by:
#   - Stage 1C V3.1 general visual verifiers
#   - Stage 1C V3.2 pairwise specialists
#
# This file contains no training code.
# =========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

from scipy.stats import entropy, skew

from skimage.color import rgb2gray, rgb2hsv, rgb2lab
from skimage.transform import resize
from skimage.feature import (
    hog,
    local_binary_pattern,
    graycomatrix,
    graycoprops,
    canny
)
from skimage.filters import sobel, laplace


warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True


# =========================================================
# Config
# =========================================================

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMG_SIZE = (224, 224)
PATCH_GRID = 4
RANDOM_STATE = 42

CATEGORY_ALIASES = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]


# =========================================================
# Optional Data Loader
# Useful only for testing, not required by FastAPI.
# =========================================================

def collect_relevant_images(data_dir):
    rows = []
    data_dir = Path(data_dir)

    for folder in data_dir.iterdir():
        if not folder.is_dir():
            continue

        raw_name = folder.name.strip()
        category = CATEGORY_ALIASES.get(raw_name, raw_name)

        if category not in CATEGORIES:
            continue

        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                rows.append({
                    "path": str(p),
                    "raw_folder": raw_name,
                    "category": category
                })

    return pd.DataFrame(rows)


# =========================================================
# Helpers
# =========================================================

def safe_hist(vals, bins, range_):
    hist, _ = np.histogram(
        vals,
        bins=bins,
        range=range_,
        density=True
    )

    hist = np.nan_to_num(
        hist,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return hist.astype(np.float32)


def color_ratio_features(img_rgb, hsv):
    r = img_rgb[:, :, 0]
    g = img_rgb[:, :, 1]
    b = img_rgb[:, :, 2]

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    blue_water = ((h > 0.50) & (h < 0.72) & (s > 0.12) & (v > 0.20)).mean()
    cyan_water = ((h > 0.42) & (h < 0.58) & (s > 0.10) & (v > 0.25)).mean()

    brown_wood = (
        ((h > 0.04) & (h < 0.16) & (s > 0.18))
        |
        ((r > g) & (g > b) & (r > 0.25))
    ).mean()

    white_wall = ((s < 0.18) & (v > 0.55)).mean()
    gray_wall = ((s < 0.12) & (v > 0.25) & (v < 0.80)).mean()

    dark_wire = (v < 0.20).mean()
    black_objects = ((v < 0.15) & (s < 0.60)).mean()

    high_saturation = (s > 0.55).mean()

    return np.array([
        blue_water,
        cyan_water,
        brown_wood,
        white_wall,
        gray_wall,
        dark_wire,
        black_objects,
        high_saturation,
        float(np.mean(r)),
        float(np.mean(g)),
        float(np.mean(b)),
        float(np.std(r)),
        float(np.std(g)),
        float(np.std(b)),
        float(np.mean(s)),
        float(np.std(s)),
        float(np.mean(v)),
        float(np.std(v)),
    ], dtype=np.float32)


def patch_features(img_rgb, gray, hsv, grid=4):
    h, w = gray.shape
    feats = []

    patch_h = h // grid
    patch_w = w // grid

    for i in range(grid):
        for j in range(grid):
            y1 = i * patch_h
            y2 = h if i == grid - 1 else (i + 1) * patch_h

            x1 = j * patch_w
            x2 = w if j == grid - 1 else (j + 1) * patch_w

            p_rgb = img_rgb[y1:y2, x1:x2, :]
            p_gray = gray[y1:y2, x1:x2]
            p_hsv = hsv[y1:y2, x1:x2, :]

            p_edges = canny(p_gray).mean()
            p_sobel = sobel(p_gray).mean()
            p_lap = laplace(p_gray).var()

            gray_hist = safe_hist(
                p_gray.ravel(),
                bins=16,
                range_=(0, 1)
            )

            p_entropy = entropy(gray_hist + 1e-8)

            feats.extend([
                float(p_gray.mean()),
                float(p_gray.std()),
                float(p_hsv[:, :, 0].mean()),
                float(p_hsv[:, :, 1].mean()),
                float(p_hsv[:, :, 2].mean()),
                float(p_edges),
                float(p_sobel),
                float(p_lap),
                float(p_entropy),
            ])

            for ch in range(3):
                vals = p_rgb[:, :, ch].ravel()

                feats.extend([
                    float(np.mean(vals)),
                    float(np.std(vals)),
                ])

    return np.array(feats, dtype=np.float32)


# =========================================================
# Main Feature Extractor
# =========================================================

def extract_stage1c_v31_features(image_path, img_size=IMG_SIZE):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_pil = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    img = np.array(img_pil)

    img = resize(
        img,
        img_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    if img.max() > 1.0:
        img /= 255.0

    gray = rgb2gray(img)
    hsv = rgb2hsv(img)
    lab = rgb2lab(img)

    features = []

    # 1) HOG
    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    features.extend(hog_feat.astype(np.float32))

    # 2) Multi-scale LBP
    lbp_1 = local_binary_pattern(
        gray,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_1_hist = safe_hist(
        lbp_1.ravel(),
        bins=10,
        range_=(0, 10)
    )

    lbp_2 = local_binary_pattern(
        gray,
        P=16,
        R=2,
        method="uniform"
    )

    lbp_2_hist = safe_hist(
        lbp_2.ravel(),
        bins=18,
        range_=(0, 18)
    )

    lbp_3 = local_binary_pattern(
        gray,
        P=24,
        R=3,
        method="uniform"
    )

    lbp_3_hist = safe_hist(
        lbp_3.ravel(),
        bins=26,
        range_=(0, 26)
    )

    features.extend(lbp_1_hist)
    features.extend(lbp_2_hist)
    features.extend(lbp_3_hist)

    # 3) RGB / HSV histograms
    for space in [img, hsv]:
        for ch in range(3):
            features.extend(
                safe_hist(
                    space[:, :, ch].ravel(),
                    bins=32,
                    range_=(0, 1)
                )
            )

    # 4) LAB histograms
    features.extend(
        safe_hist(
            lab[:, :, 0].ravel(),
            bins=32,
            range_=(0, 100)
        )
    )

    features.extend(
        safe_hist(
            lab[:, :, 1].ravel(),
            bins=32,
            range_=(-128, 128)
        )
    )

    features.extend(
        safe_hist(
            lab[:, :, 2].ravel(),
            bins=32,
            range_=(-128, 128)
        )
    )

    # 5) Color moments
    for space in [img, hsv]:
        for ch in range(3):
            vals = space[:, :, ch].ravel()

            features.extend([
                float(np.mean(vals)),
                float(np.std(vals)),
                float(skew(vals)),
            ])

    for ch in range(3):
        vals = lab[:, :, ch].ravel()

        features.extend([
            float(np.mean(vals)),
            float(np.std(vals)),
            float(skew(vals)),
        ])

    # 6) GLCM texture
    gray_u8 = np.clip(gray * 63, 0, 63).astype(np.uint8)

    glcm = graycomatrix(
        gray_u8,
        distances=[1, 2, 4],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=64,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM"
    ]:
        features.extend(
            graycoprops(glcm, prop).ravel().astype(np.float32)
        )

    # 7) Edge / sharpness / entropy
    edges = canny(gray)
    edge_density = edges.mean()

    sob = sobel(gray)

    sobel_hist = safe_hist(
        sob.ravel(),
        bins=24,
        range_=(0, 1)
    )

    lap_var = laplace(gray).var()

    gray_hist = safe_hist(
        gray.ravel(),
        bins=64,
        range_=(0, 1)
    )

    ent = entropy(gray_hist + 1e-8)

    features.extend(sobel_hist)

    features.extend([
        float(edge_density),
        float(lap_var),
        float(ent),
        float(orig_w / (orig_h + 1e-8)),
        float(gray.mean()),
        float(gray.std()),
        float(hsv[:, :, 0].mean()),
        float(hsv[:, :, 0].std()),
        float(hsv[:, :, 1].mean()),
        float(hsv[:, :, 1].std()),
        float(hsv[:, :, 2].mean()),
        float(hsv[:, :, 2].std()),
        float(orig_w),
        float(orig_h),
    ])

    # 8) Domain-aware features
    features.extend(
        color_ratio_features(img, hsv)
    )

    # 9) Patch-level features
    features.extend(
        patch_features(
            img,
            gray,
            hsv,
            grid=PATCH_GRID
        )
    )

    feat = np.array(
        features,
        dtype=np.float32
    )

    feat = np.nan_to_num(
        feat,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return feat
'''

output_path = INFERENCE_DIR / "stage1c_v31_features.py"
output_path.write_text(code, encoding="utf-8")

print("✅ Saved:", output_path)
print("Exists:", output_path.exists())

✅ Saved: maintenance_ai_production_clean/inference/stage1c_v31_features.py
Exists: True


In [11]:
import py_compile
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/stage1c_v31_features.py")

py_compile.compile(str(path), doraise=True)
print("✅ stage1c_v31_features.py syntax OK")

✅ stage1c_v31_features.py syntax OK


In [12]:
import sys
import importlib
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())
if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

if "inference.stage1c_v31_features" in sys.modules:
    del sys.modules["inference.stage1c_v31_features"]

importlib.invalidate_caches()

from inference.stage1c_v31_features import extract_stage1c_v31_features

print("✅ stage1c_v31_features imported successfully")

✅ stage1c_v31_features imported successfully


In [13]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"

code = r'''
# =========================================================
# Stage 1C Production Inference
# Text-Guided Visual Category Verifier
#
# Input:
#   image_path + text_category from Stage 2
#
# Output:
#   VERIFIED / NOT_VERIFIED
# =========================================================

import warnings
from pathlib import Path

import joblib
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif

try:
    from .stage1c_v31_features import (
        extract_stage1c_v31_features,
        IMG_SIZE,
        CATEGORIES
    )
except ImportError:
    from stage1c_v31_features import (
        extract_stage1c_v31_features,
        IMG_SIZE,
        CATEGORIES
    )


warnings.filterwarnings("ignore")


# =========================================================
# Joblib Required Classes
# =========================================================

class Stage1CV31Preprocessor:
    def __init__(self, k_best=4500):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


class Stage1CV32PairwisePreprocessor:
    def __init__(self, k_best=5000):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# Config
# =========================================================

VALID_CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]

CATEGORY_ALIASES = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

FINAL_VERIFY_THRESHOLD = 0.55
FINAL_STRONG_VERIFY_THRESHOLD = 0.65
FINAL_REJECT_THRESHOLD = 0.42

GENERAL_VERIFIER_WEIGHT = 0.40
MULTICLASS_WEIGHT = 0.25
PAIRWISE_WEIGHT = 0.35


# =========================================================
# Utilities
# =========================================================

def normalize_category(category):
    category = str(category).strip()
    category = CATEGORY_ALIASES.get(category, category)

    if category not in VALID_CATEGORIES:
        raise ValueError(
            f"Unknown category: {category}. "
            f"Expected one of: {VALID_CATEGORIES}"
        )

    return category


def safe_predict_binary_proba(models, X, weights=None):
    if weights is None:
        weights = {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }

    total = 0.0

    for name, weight in weights.items():
        model = models[name]

        if hasattr(model, "predict_proba"):
            p = float(model.predict_proba(X)[0, 1])
        elif hasattr(model, "decision_function"):
            score = float(model.decision_function(X)[0])
            p = float(1.0 / (1.0 + np.exp(-score)))
        else:
            p = float(model.predict(X)[0])

        total += weight * p

    return float(total)


def safe_predict_multiclass_proba(models, X, weights=None):
    if weights is None:
        weights = {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }

    final_probs = None

    for name, weight in weights.items():
        model = models[name]

        if not hasattr(model, "predict_proba"):
            raise ValueError(f"Multiclass model {name} has no predict_proba().")

        probs = model.predict_proba(X)

        if final_probs is None:
            final_probs = weight * probs
        else:
            final_probs += weight * probs

    return final_probs[0]


def pair_key(cat_a, cat_b):
    return f"{cat_a}__vs__{cat_b}"


def find_pairwise_specialist(pairwise_bundle, text_category, other_category):
    specialists = pairwise_bundle["pairwise_specialists"]

    key_forward = pair_key(text_category, other_category)
    key_reverse = pair_key(other_category, text_category)

    if key_forward in specialists:
        return specialists[key_forward], "forward"

    if key_reverse in specialists:
        return specialists[key_reverse], "reverse"

    return None, None


# =========================================================
# Load
# =========================================================

def load_stage1c(artifacts_dir=None):
    if artifacts_dir is None:
        artifacts_dir = Path(__file__).resolve().parents[1] / "artifacts" / "stage1c"
    else:
        artifacts_dir = Path(artifacts_dir)

    v31_bundle_path = artifacts_dir / "stage1c_v31_bundle.joblib"
    v32_bundle_path = artifacts_dir / "stage1c_v32_pairwise_bundle.joblib"

    if not v31_bundle_path.exists():
        raise FileNotFoundError(f"Missing Stage 1C V3.1 bundle: {v31_bundle_path}")

    if not v32_bundle_path.exists():
        raise FileNotFoundError(f"Missing Stage 1C V3.2 bundle: {v32_bundle_path}")

    v31_bundle = joblib.load(v31_bundle_path)
    v32_bundle = joblib.load(v32_bundle_path)

    required_v31 = [
        "multiclass_classifier",
        "verifiers",
        "categories",
        "img_size",
        "feature_extractor_name"
    ]

    required_v32 = [
        "pairwise_specialists",
        "categories",
        "pairs",
        "feature_extractor_name"
    ]

    missing_v31 = [k for k in required_v31 if k not in v31_bundle]
    missing_v32 = [k for k in required_v32 if k not in v32_bundle]

    if missing_v31:
        raise KeyError(f"Missing keys in Stage 1C V3.1 bundle: {missing_v31}")

    if missing_v32:
        raise KeyError(f"Missing keys in Stage 1C V3.2 bundle: {missing_v32}")

    return {
        "v31": v31_bundle,
        "v32": v32_bundle,
        "categories": VALID_CATEGORIES
    }


# =========================================================
# Score Functions
# =========================================================

def predict_general_verifier_score(X_raw, text_category, stage1c):
    verifier = stage1c["v31"]["verifiers"][text_category]

    preprocessor = verifier["preprocessor"]
    models = verifier["models"]

    weights = verifier.get(
        "ensemble_weights",
        {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }
    )

    X = preprocessor.transform(X_raw)

    return safe_predict_binary_proba(
        models=models,
        X=X,
        weights=weights
    )


def predict_multiclass_category_score(X_raw, text_category, stage1c):
    classifier = stage1c["v31"]["multiclass_classifier"]

    preprocessor = classifier["preprocessor"]
    models = classifier["models"]
    label_encoder = classifier["label_encoder"]

    weights = classifier.get(
        "ensemble_weights",
        {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }
    )

    X = preprocessor.transform(X_raw)

    probs = safe_predict_multiclass_proba(
        models=models,
        X=X,
        weights=weights
    )

    classes = list(label_encoder.classes_)

    if text_category not in classes:
        raise ValueError(
            f"Category {text_category} not found in multiclass classes: {classes}"
        )

    cat_idx = classes.index(text_category)
    category_prob = float(probs[cat_idx])

    top_idx = int(np.argmax(probs))
    top_category = classes[top_idx]
    top_prob = float(probs[top_idx])

    all_probs = {
        str(cls): float(probs[i])
        for i, cls in enumerate(classes)
    }

    return category_prob, top_category, top_prob, all_probs


def predict_pairwise_support_scores(X_raw, text_category, stage1c):
    v32 = stage1c["v32"]

    pairwise_details = []
    support_scores = []
    support_votes = 0

    other_categories = [
        c for c in VALID_CATEGORIES
        if c != text_category
    ]

    for other_category in other_categories:
        specialist, direction = find_pairwise_specialist(
            pairwise_bundle=v32,
            text_category=text_category,
            other_category=other_category
        )

        if specialist is None:
            continue

        preprocessor = specialist["preprocessor"]
        models = specialist["models"]

        X = preprocessor.transform(X_raw)

        prob_cat_a = safe_predict_binary_proba(
            models=models,
            X=X,
            weights={
                "xgb": 0.45,
                "extra_trees": 0.35,
                "random_forest": 0.20
            }
        )

        if direction == "forward":
            support_prob = prob_cat_a
            pair_orientation = f"{text_category}__vs__{other_category}"
        else:
            support_prob = 1.0 - prob_cat_a
            pair_orientation = f"{other_category}__vs__{text_category}"

        support_prob = float(support_prob)
        support_scores.append(support_prob)

        if support_prob >= 0.50:
            support_votes += 1

        pairwise_details.append({
            "pair": pair_orientation,
            "text_category": text_category,
            "other_category": other_category,
            "direction": direction,
            "support_prob": support_prob
        })

    if len(support_scores) == 0:
        return 0.0, 0.0, 0, pairwise_details

    return (
        float(np.mean(support_scores)),
        float(np.min(support_scores)),
        int(support_votes),
        pairwise_details
    )


# =========================================================
# Decision
# =========================================================

def decide_stage1c(
    final_score,
    general_score,
    multiclass_score,
    pairwise_mean,
    pairwise_min,
    pairwise_votes,
    multiclass_top_category,
    text_category
):
    if (
        final_score >= FINAL_STRONG_VERIFY_THRESHOLD
        and pairwise_votes >= 2
        and general_score >= 0.45
    ):
        return "VERIFIED", "strong_visual_support"

    if (
        final_score >= FINAL_VERIFY_THRESHOLD
        and pairwise_votes >= 2
        and pairwise_mean >= 0.50
    ):
        return "VERIFIED", "verified_by_combined_score"

    if (
        multiclass_top_category == text_category
        and multiclass_score >= 0.55
        and pairwise_votes >= 2
        and final_score >= 0.50
    ):
        return "VERIFIED", "verified_by_multiclass_agreement"

    if final_score <= FINAL_REJECT_THRESHOLD:
        return "NOT_VERIFIED", "low_final_score"

    if pairwise_votes <= 1 and pairwise_mean < 0.50:
        return "NOT_VERIFIED", "pairwise_disagreement"

    if multiclass_top_category != text_category and multiclass_score < 0.25:
        return "NOT_VERIFIED", "multiclass_disagreement"

    return "NOT_VERIFIED", "not_enough_visual_evidence"


# =========================================================
# Main Prediction
# =========================================================

def predict_stage1c_visual_category(
    image_path,
    text_category,
    stage1c,
    return_details=True
):
    image_path = Path(image_path)
    text_category = normalize_category(text_category)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    features = extract_stage1c_v31_features(
        image_path=image_path,
        img_size=IMG_SIZE
    )

    X_raw = features.reshape(1, -1)

    general_score = predict_general_verifier_score(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    (
        multiclass_score,
        multiclass_top_category,
        multiclass_top_prob,
        multiclass_all_probs
    ) = predict_multiclass_category_score(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    (
        pairwise_mean,
        pairwise_min,
        pairwise_votes,
        pairwise_details
    ) = predict_pairwise_support_scores(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    final_score = float(
        GENERAL_VERIFIER_WEIGHT * general_score
        +
        MULTICLASS_WEIGHT * multiclass_score
        +
        PAIRWISE_WEIGHT * pairwise_mean
    )

    decision, reason = decide_stage1c(
        final_score=final_score,
        general_score=general_score,
        multiclass_score=multiclass_score,
        pairwise_mean=pairwise_mean,
        pairwise_min=pairwise_min,
        pairwise_votes=pairwise_votes,
        multiclass_top_category=multiclass_top_category,
        text_category=text_category
    )

    result = {
        "stage": "Stage 1C",
        "image_path": str(image_path),
        "text_category": text_category,
        "decision": decision,
        "confidence": final_score,
        "reason": reason,
        "general_score": float(general_score),
        "multiclass_score": float(multiclass_score),
        "multiclass_top_category": multiclass_top_category,
        "multiclass_top_probability": float(multiclass_top_prob),
        "pairwise_mean": float(pairwise_mean),
        "pairwise_min": float(pairwise_min),
        "pairwise_votes": int(pairwise_votes),
    }

    if return_details:
        result["multiclass_all_probs"] = multiclass_all_probs
        result["pairwise_details"] = pairwise_details

    return result
'''

output_path = INFERENCE_DIR / "stage1c_visual_verifier.py"
output_path.write_text(code, encoding="utf-8")

print("✅ Saved:", output_path)
print("Exists:", output_path.exists())

✅ Saved: maintenance_ai_production_clean/inference/stage1c_visual_verifier.py
Exists: True


In [14]:
import py_compile
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/stage1c_visual_verifier.py")

py_compile.compile(str(path), doraise=True)
print("✅ stage1c_visual_verifier.py syntax OK")

✅ stage1c_visual_verifier.py syntax OK


In [16]:
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/stage1c_visual_verifier.py")

text = path.read_text(encoding="utf-8")

old = '''
def load_stage1c(artifacts_dir=None):
    if artifacts_dir is None:
        artifacts_dir = Path(__file__).resolve().parents[1] / "artifacts" / "stage1c"
    else:
        artifacts_dir = Path(artifacts_dir)
'''

new = '''
def load_stage1c(artifacts_dir=None):
    # -----------------------------------------------------
    # Compatibility fix:
    # The Stage 1C bundles were saved from a notebook, so
    # joblib may look for these classes under __main__.
    # We register them there before loading.
    # -----------------------------------------------------
    import __main__

    setattr(__main__, "Stage1CV31Preprocessor", Stage1CV31Preprocessor)
    setattr(__main__, "Stage1CV32PairwisePreprocessor", Stage1CV32PairwisePreprocessor)

    if artifacts_dir is None:
        artifacts_dir = Path(__file__).resolve().parents[1] / "artifacts" / "stage1c"
    else:
        artifacts_dir = Path(artifacts_dir)
'''

if old not in text:
    print("❌ Could not find target block. No changes made.")
else:
    text = text.replace(old, new)
    path.write_text(text, encoding="utf-8")
    print("✅ Patched stage1c_visual_verifier.py")

✅ Patched stage1c_visual_verifier.py


In [17]:
import sys
import importlib
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())
if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

for m in [
    "inference.stage1c_v31_features",
    "inference.stage1c_visual_verifier"
]:
    if m in sys.modules:
        del sys.modules[m]

importlib.invalidate_caches()

from inference.stage1c_visual_verifier import (
    load_stage1c,
    predict_stage1c_visual_category
)

stage1c = load_stage1c()
print("✅ Stage 1C loaded from clean package")

test_image = "Fake Image Detection/relevant/سباكه/photo_40_2025-12-01_18-05-45.jpg"

result = predict_stage1c_visual_category(
    image_path=test_image,
    text_category="سباكة",
    stage1c=stage1c,
    return_details=False
)

print(result["decision"])
print(round(result["confidence"], 4))
print(result["reason"])

sh: line 1: nvidia-smi: command not found


✅ Stage 1C loaded from clean package
VERIFIED
0.9319
strong_visual_support


In [25]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")
INFERENCE_DIR = BASE_DIR / "inference"

code = r'''
# =========================================================
# Final Production Decision Engine
#
# Full Pipeline:
#   image_path + Arabic description
#       ↓
#   Stage 1A: image relevance
#       ↓
#   Stage 2: text category/subcategory
#       ↓
#   Stage 1C: visual category verification
#       ↓
#   Final decision: MATCH / MISMATCH
# =========================================================

from pathlib import Path

try:
    from .stage1a_inference import (
        load_stage1a,
        predict_stage1a_image
    )

    from .stage2_inference import (
        load_stage2,
        predict_stage2_text
    )

    from .stage1c_visual_verifier import (
        load_stage1c,
        predict_stage1c_visual_category
    )

except ImportError:
    from stage1a_inference import (
        load_stage1a,
        predict_stage1a_image
    )

    from stage2_inference import (
        load_stage2,
        predict_stage2_text
    )

    from stage1c_visual_verifier import (
        load_stage1c,
        predict_stage1c_visual_category
    )


# =========================================================
# Load Full Pipeline
# =========================================================

def load_full_pipeline():
    """
    Load all production stages once.
    In FastAPI, call this once at startup, not per request.
    """

    print("Loading Stage 1A...")
    stage1a_bundle, stage1a_config = load_stage1a()

    print("Loading Stage 2...")
    stage2 = load_stage2()

    print("Loading Stage 1C...")
    stage1c = load_stage1c()

    print("✅ Full pipeline loaded successfully.")

    return {
        "stage1a_bundle": stage1a_bundle,
        "stage1a_config": stage1a_config,
        "stage2": stage2,
        "stage1c": stage1c
    }


# =========================================================
# Final Decision Logic
# =========================================================

def make_final_decision(stage1a_result, stage2_result, stage1c_result=None):
    """
    Final production decision:
    MATCH / MISMATCH only.

    Strict path:
        Stage 1A RELEVANT + Stage 2 MATCH + Stage 1C VERIFIED => MATCH

    Soft visual fallback:
        Used for real mobile images where Stage 1C is conservative but
        pairwise visual evidence still supports the text category.
    """

    if stage1a_result["decision"] != "RELEVANT":
        return "MISMATCH", "image_not_relevant"

    if stage2_result["decision"] != "MATCH":
        return "MISMATCH", "text_not_matched"

    if stage1c_result is None:
        return "MISMATCH", "missing_visual_verification"

    if stage1c_result["decision"] == "VERIFIED":
        return "MATCH", "image_and_text_category_match"

    # -----------------------------------------------------
    # Soft visual fallback for real-world mobile images
    # -----------------------------------------------------
    stage1a_conf = float(stage1a_result.get("prob_relevant", 0.0))
    stage2_rule = float(stage2_result.get("rule_score", 0.0))
    stage2_conf = float(stage2_result.get("confidence", 0.0))

    visual_conf = float(stage1c_result.get("confidence", 0.0))
    pairwise_mean = float(stage1c_result.get("pairwise_mean", 0.0))
    pairwise_votes = int(stage1c_result.get("pairwise_votes", 0))
    multiclass_score = float(stage1c_result.get("multiclass_score", 0.0))

    if (
        stage1a_conf >= 0.70
        and stage2_conf >= 0.45
        and stage2_rule >= 0.80
        and visual_conf >= 0.34
        and pairwise_votes >= 2
        and pairwise_mean >= 0.50
    ):
        return "MATCH", "soft_visual_accept_real_world_image"

    # Slightly weaker text confidence but stronger pairwise support
    if (
        stage1a_conf >= 0.70
        and stage2_rule >= 0.90
        and visual_conf >= 0.32
        and pairwise_votes >= 2
        and pairwise_mean >= 0.53
        and multiclass_score >= 0.20
    ):
        return "MATCH", "soft_visual_accept_pairwise_support"

    return "MISMATCH", "image_category_does_not_match_text_category"


# =========================================================
# Main Prediction
# =========================================================

def predict_request(
    image_path,
    arabic_description,
    pipeline,
    deploy_reject_threshold=0.15,
    return_details=True
):
    """
    Full production prediction.

    Args:
        image_path:
            Path to user image.

        arabic_description:
            Arabic problem description from user.

        pipeline:
            Loaded object from load_full_pipeline().

        deploy_reject_threshold:
            Runtime reject threshold for Stage 1A.

        return_details:
            If True, return full stage outputs.
            If False, return compact output.

    Returns:
        dict
    """

    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # -----------------------------------------------------
    # Stage 1A: Image relevance
    # -----------------------------------------------------
    stage1a_result = predict_stage1a_image(
        image_path=image_path,
        stage1_bundle=pipeline["stage1a_bundle"],
        deploy_reject_threshold=deploy_reject_threshold,
        use_deploy_reject_threshold=True
    )

    if stage1a_result["decision"] != "RELEVANT":
        final_decision, reason = make_final_decision(
            stage1a_result=stage1a_result,
            stage2_result={
                "decision": "SKIPPED",
                "category": None,
                "subcategory": None
            },
            stage1c_result=None
        )

        compact = {
            "final_decision": final_decision,
            "reason": reason,
            "image_path": str(image_path),
            "arabic_description": arabic_description,
            "image_relevance": stage1a_result["decision"],
            "text_decision": "SKIPPED",
            "text_category": None,
            "text_subcategory": None,
            "visual_verification": "SKIPPED"
        }

        if not return_details:
            return compact

        compact["stage1a"] = stage1a_result
        compact["stage2"] = {
            "stage": "Stage 2",
            "decision": "SKIPPED",
            "reason": "image_not_relevant"
        }
        compact["stage1c"] = {
            "stage": "Stage 1C",
            "decision": "SKIPPED",
            "reason": "image_not_relevant"
        }

        return compact

    # -----------------------------------------------------
    # Stage 2: Arabic text category/subcategory
    # -----------------------------------------------------
    stage2_result = predict_stage2_text(
        user_text=arabic_description,
        stage2=pipeline["stage2"]
    )

    if stage2_result["decision"] != "MATCH":
        final_decision, reason = make_final_decision(
            stage1a_result=stage1a_result,
            stage2_result=stage2_result,
            stage1c_result=None
        )

        compact = {
            "final_decision": final_decision,
            "reason": reason,
            "image_path": str(image_path),
            "arabic_description": arabic_description,
            "image_relevance": stage1a_result["decision"],
            "text_decision": stage2_result["decision"],
            "text_category": stage2_result.get("category"),
            "text_subcategory": stage2_result.get("subcategory"),
            "visual_verification": "SKIPPED"
        }

        if not return_details:
            return compact

        compact["stage1a"] = stage1a_result
        compact["stage2"] = stage2_result
        compact["stage1c"] = {
            "stage": "Stage 1C",
            "decision": "SKIPPED",
            "reason": "text_not_matched"
        }

        return compact

    # -----------------------------------------------------
    # Stage 1C: Visual category verification
    # -----------------------------------------------------
    text_category = stage2_result["category"]

    stage1c_result = predict_stage1c_visual_category(
        image_path=image_path,
        text_category=text_category,
        stage1c=pipeline["stage1c"],
        return_details=return_details
    )

    # -----------------------------------------------------
    # Final decision
    # -----------------------------------------------------
    final_decision, reason = make_final_decision(
        stage1a_result=stage1a_result,
        stage2_result=stage2_result,
        stage1c_result=stage1c_result
    )

    compact = {
        "final_decision": final_decision,
        "reason": reason,
        "image_path": str(image_path),
        "arabic_description": arabic_description,
        "image_relevance": stage1a_result["decision"],
        "image_relevance_confidence": stage1a_result.get("prob_relevant"),
        "text_decision": stage2_result["decision"],
        "text_category": stage2_result.get("category"),
        "text_subcategory": stage2_result.get("subcategory"),
        "text_confidence": stage2_result.get("confidence"),
        "visual_verification": stage1c_result["decision"],
        "visual_confidence": stage1c_result.get("confidence")
    }

    if not return_details:
        return compact

    compact["stage1a"] = stage1a_result
    compact["stage2"] = stage2_result
    compact["stage1c"] = stage1c_result

    return compact


# =========================================================
# Pretty Print
# =========================================================

def print_final_result(result):
    print("\n" + "=" * 70)
    print("FINAL DECISION:", result["final_decision"])
    print("REASON:", result["reason"])
    print("=" * 70)

    print("\nInput:")
    print("Image:", result["image_path"])
    print("Description:", result["arabic_description"])

    print("\nStage 1A — Image Relevance:")
    print("Decision:", result.get("image_relevance"))
    if result.get("image_relevance_confidence") is not None:
        print("Confidence:", round(result["image_relevance_confidence"], 4))

    print("\nStage 2 — Arabic Text:")
    print("Decision:", result.get("text_decision"))
    print("Category:", result.get("text_category"))
    print("Subcategory:", result.get("text_subcategory"))
    if result.get("text_confidence") is not None:
        print("Confidence:", round(result["text_confidence"], 4))

    print("\nStage 1C — Visual Category Verification:")
    print("Decision:", result.get("visual_verification"))
    if result.get("visual_confidence") is not None:
        print("Confidence:", round(result["visual_confidence"], 4))


# =========================================================
# Optional Local Test
# =========================================================

def test_decision_engine():
    pipeline = load_full_pipeline()

    test_cases = [
        {
            "image_path": "Fake Image Detection/relevant/سباكه/photo_40_2025-12-01_18-05-45.jpg",
            "description": "الحنفية بتسرب ميه طول الوقت"
        },
        {
            "image_path": "Fake Image Detection/relevant/كهربا/photo_17_2025-12-01_15-46-05.jpg",
            "description": "فيشة الكهرباء بتطلع شرار"
        },
        {
            "image_path": "Fake Image Detection/relevant/نقاشه/photo_58_2025-12-03_18-18-15.jpg",
            "description": "في شروخ واضحة في الحيطة"
        },
        {
            # deliberate mismatch: plumbing image + electricity text
            "image_path": "Fake Image Detection/relevant/سباكه/photo_40_2025-12-01_18-05-45.jpg",
            "description": "فيشة الكهرباء بتطلع شرار"
        }
    ]

    for case in test_cases:
        image_path = case["image_path"]
        description = case["description"]

        if not Path(image_path).exists():
            print("\n⚠️ Skipping missing image:", image_path)
            continue

        result = predict_request(
            image_path=image_path,
            arabic_description=description,
            pipeline=pipeline,
            return_details=False
        )

        print_final_result(result)


if __name__ == "__main__":
    test_decision_engine()
'''

output_path = INFERENCE_DIR / "decision_engine.py"
output_path.write_text(code, encoding="utf-8")

print("✅ Saved:", output_path)
print("Exists:", output_path.exists())

✅ Saved: maintenance_ai_production_clean/inference/decision_engine.py
Exists: True


In [26]:
import py_compile
from pathlib import Path

path = Path("maintenance_ai_production_clean/inference/decision_engine.py")

py_compile.compile(str(path), doraise=True)
print("✅ decision_engine.py syntax OK")

✅ decision_engine.py syntax OK


In [33]:
import sys
import importlib
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())
if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

for m in [
    "inference.stage1a_inference",
    "inference.stage2_inference",
    "inference.stage1c_v31_features",
    "inference.stage1c_visual_verifier",
    "inference.decision_engine",
]:
    if m in sys.modules:
        del sys.modules[m]

importlib.invalidate_caches()

from inference.decision_engine import (
    load_full_pipeline,
    predict_request,
    print_final_result
)

pipeline = load_full_pipeline()

result = predict_request(
    image_path="WhatsApp Image 2026-05-08 at 1.13.31 AM (2).jpeg",
    arabic_description="عداد الكهربا ولع",
    pipeline=pipeline,
    return_details=True
)

print_final_result(result)
print("\nReason:", result["reason"])

Loading Stage 1A...
Loading Stage 2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Stage 1C...
✅ Full pipeline loaded successfully.

FINAL DECISION: MATCH
REASON: image_and_text_category_match

Input:
Image: WhatsApp Image 2026-05-08 at 1.13.31 AM (2).jpeg
Description: عداد الكهربا ولع

Stage 1A — Image Relevance:
Decision: RELEVANT
Confidence: 0.9977

Stage 2 — Arabic Text:
Decision: MATCH
Category: كهرباء
Subcategory: عطل كهرباء عام
Confidence: 0.4439

Stage 1C — Visual Category Verification:
Decision: VERIFIED
Confidence: 0.9163

Reason: image_and_text_category_match


In [34]:
from inference.decision_engine import load_full_pipeline, predict_request

pipeline = load_full_pipeline()

result = predict_request(
    image_path="WhatsApp Image 2026-05-08 at 1.13.30 AM (2).jpeg",
    arabic_description="الحنفية بتسرب ميه طول الوقت",
    pipeline=pipeline,
    return_details=False
)

Loading Stage 1A...
Loading Stage 2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Stage 1C...
✅ Full pipeline loaded successfully.


In [35]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")

requirements = """
numpy
pandas
scipy
scikit-learn
scikit-image
Pillow
joblib
xgboost
sentence-transformers
torch
transformers
tqdm
fastapi
uvicorn
python-multipart
"""

(BASE_DIR / "requirements.txt").write_text(requirements.strip() + "\n", encoding="utf-8")

print("✅ Saved:", BASE_DIR / "requirements.txt")

✅ Saved: maintenance_ai_production_clean/requirements.txt


In [37]:
from pathlib import Path

BASE_DIR = Path("maintenance_ai_production_clean")

readme = r"""
# Maintenance AI Production Package

This package contains the clean production inference pipeline for the maintenance image/text matching system.

## Purpose

The system receives:

- An image path
- An Arabic user description

Then returns:

- `MATCH`
- `MISMATCH`

The final decision is based on three stages:

1. **Stage 1A — Image Relevance**
   - Checks whether the uploaded image is maintenance-related.
   - Output: `RELEVANT`, `IRRELEVANT`, or `UNCERTAIN`.

2. **Stage 2 — Arabic Text Understanding**
   - Classifies the user description into:
     - category
     - subcategory
   - Output examples:
     - `سباكة / تسريب مياه`
     - `كهرباء / عطل كهرباء عام`
     - `نقاشة / تشققات جدار`
     - `نجارة / عطل باب`

3. **Stage 1C — Visual Category Verification**
   - Checks whether the image visually supports the category predicted from the text.
   - Output:
     - `VERIFIED`
     - `NOT_VERIFIED`

The final engine returns `MATCH` only when the image and text are consistent.

---

## Folder Structure

```text
maintenance_ai_production_clean/
│
├── artifacts/
│   ├── stage1a/
│   │   ├── stage1_bundle.joblib
│   │   └── config.json
│   │
│   ├── stage2/
│   │   ├── stage2_best_model.joblib
│   │   ├── stage2_word_tfidf.joblib
│   │   ├── stage2_char_tfidf.joblib
│   │   ├── stage2_feature_names.joblib
│   │   ├── stage2_binary_threshold.joblib
│   │   ├── stage2_accept_threshold.joblib
│   │   ├── stage2_reject_threshold.joblib
│   │   └── stage2_clean_reference_dataset.csv
│   │
│   └── stage1c/
│       ├── stage1c_v31_bundle.joblib
│       └── stage1c_v32_pairwise_bundle.joblib
│
├── inference/
│   ├── stage1a_inference.py
│   ├── stage2_inference.py
│   ├── stage1c_v31_features.py
│   ├── stage1c_visual_verifier.py
│   └── decision_engine.py
│
├── requirements.txt
└── README.md
"""

In [38]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Note: you may need to restart the kernel to use updated packages.


In [40]:
import sys
from pathlib import Path

pkg_path = str(Path("maintenance_ai_production_clean").resolve())

if pkg_path not in sys.path:
    sys.path.insert(0, pkg_path)

from inference.decision_engine import load_full_pipeline, predict_request

pipeline = load_full_pipeline()

result = predict_request(
    image_path="00a4c4999dc0c6a442a9318d813b8dd0.jpg",
    arabic_description="الحنفية بتسرب ميه طول الوقت",
    pipeline=pipeline,
    return_details=False
)

print(result)

Loading Stage 1A...
Loading Stage 2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Stage 1C...
✅ Full pipeline loaded successfully.
{'final_decision': 'MISMATCH', 'reason': 'image_category_does_not_match_text_category', 'image_path': '00a4c4999dc0c6a442a9318d813b8dd0.jpg', 'arabic_description': 'الحنفية بتسرب ميه طول الوقت', 'image_relevance': 'RELEVANT', 'image_relevance_confidence': 0.9662116255460477, 'text_decision': 'MATCH', 'text_category': 'سباكة', 'text_subcategory': 'تسريب مياه', 'text_confidence': 0.4985301150802594, 'visual_verification': 'NOT_VERIFIED', 'visual_confidence': 0.292723217700277}


In [6]:
import shutil
from pathlib import Path

folder_path = Path("stage 1c")

if not folder_path.exists():
    raise FileNotFoundError("Folder not found: maintenance_ai_production_clean")

zip_path = shutil.make_archive(
    base_name="stage 1c",
    format="zip",
    root_dir=folder_path.parent,
    base_dir=folder_path.name
)

print("ZIP created successfully:")
print(zip_path)

ZIP created successfully:
/mmfs1/home/mohamed.salem/firstJob/pr./FID/STAGE 1 & 2/stage 1c.zip
